<div class="alert alert-info" markdown="1">

#### Homework 1, Problem 6 Supplemental Notebook

# Mathematical Foundations

### Math 124: Vectors, Matrices, and Applications, Fall 2026

<small><a style="text-decoration: none" href="https://math124.org">math124.org</a></small>
    
</div>


### Instructions

Most homeworks and some labs will have a Jupyter Notebook, like this one, containing Python code that supplements our understanding of the relevant mathematical ideas of the week.

---

To open this notebook, click the Google Colab link provided in Homework 1, Problem 6. Instructions on how to use Google Colab are at [math124.org/running-code](https://math124.org/running-code).

**Most importantly, make sure to click "Copy to Drive" and rename the notebook before proceeding!**

---

Welcome to your first programming homework question. There are a total of TODO parts, all of which together constitute your score for Problem 6 on Homework 1.

You won't need to submit this notebook anywhere. Instead, at the bottom, we'll ask you to include screenshots of specific things in your PDF submission for Homework 1 that you upload to Pensive.

## Defining functions

---

In Lab 1, we showed you how to use (or "call") existing functions, either those built into Python or those imported from other modules. You can also define your own Python functions. Think of these as recipes that can be used to perform similar logic repeatedly, without needing to copy-paste code. We'll show you how they work through an example.

Suppose we wanted a function that told us the number of years between 2026 (the current year) and any other given year. There is no built-in Python function that does this, so we create our own. Here's how we might do so. Run the cell below.

In [ ]:
def years_since(year):
    return 2026 - year

Above,
- `def` is how we tell Python we want to **define** our own function.
- `years_since` is the name of our new function.
- `year` is the function's **argument**. Think of this as a placeholder for the input that we will give to the function when we use it.
- `return` is how we tell Python what we want the function to output.

Now, as long as we've run the cell above, we can use `years_since`!

In [ ]:
years_since(1998)

In [ ]:
years_since(1776)

In [ ]:
years_since(-3)

This example function only accepted one argument. Functions can accept more than one argument too. Here's another example.

In [ ]:
def num_in_range(a, b):
    biggest = max(a, b)
    smallest = min(a, b)
    return biggest - smallest + 1

In [ ]:
num_in_range(2, 5)

In [ ]:
num_in_range(4, -10)

The function above also (1) used other existing functions in its definition, and (2) had its definition spread out over multiple lines. This is common practice.

With this in mind, you're ready to progress to the main activity.

## Images and filters

---

### Images

An image is a grid of colored squares, called pixels. The color of a pixel can be described in various ways. One common model is the RGB model, which describes a color using three numbers:
- how much **R**ed it has,
- how much **G**reen it has, and
- how much **B**lue it has.

Each of these three numbers ranges from `0` (none at all) to `255` (as much as possible). So `(255, 0, 0)` is pure red, `(0, 0, 0)` is black, `(255, 255, 255)` is white, and `(30, 60, 90)` is a muted navy.

To visualize colors in RGB format, open a new Google tab and search `rgb(15, 20, 30)`. That will show you a dark gray color, and will launch Google's built-in color palette. This will allow you to pick other colors from a spectrum and see their RGB representations.

For our purposes, an **image filter** is a function that takes a pixel's three color values and returns three new numbers. Apply that rule to every pixel in the grid and you've filtered the whole picture.!

Run the cell below to define a few helper functions that allow us to visualize various image filters. It is very long, but you don't need to understand any of the code in it.

In [ ]:
#@title Image and filter helpers { display-mode: "form" }
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen

import ipywidgets as widgets
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import clear_output, display
from PIL import Image, ImageOps
from plotly.subplots import make_subplots


RGB_HOVER = (
    "column=%{x}<br>row=%{y}<br>"
    "R=%{customdata[0]}<br>G=%{customdata[1]}<br>"
    "B=%{customdata[2]}<extra></extra>"
)


def as_rgb_array(pixels):
    arr = np.asarray(pixels)
    if arr.ndim != 3 or arr.shape[2] != 3:
        raise ValueError("Expected an image with shape (height, width, 3).")
    if np.issubdtype(arr.dtype, np.floating) and arr.size and arr.max() <= 1:
        arr = arr * 255
    return np.clip(arr, 0, 255).astype(np.uint8)


def load_pixels(filename):
    if str(filename).startswith(("https://", "http://")):
        with urlopen(filename) as response:
            source = BytesIO(response.read())
    else:
        source = filename
    with Image.open(source) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        return np.array(image)


def plotly_image_trace(pixels, name=None):
    arr = as_rgb_array(pixels)
    trace = px.imshow(arr, binary_string=False).data[0]
    trace.customdata = arr
    trace.hovertemplate = RGB_HOVER
    trace.name = name
    return trace


def show_pixels(pixels, title=None):
    fig = go.Figure(plotly_image_trace(pixels))
    fig.update_xaxes(visible=False, showgrid=False, zeroline=False)
    fig.update_yaxes(visible=False, showgrid=False, zeroline=False)
    fig.update_layout(
        title=title, paper_bgcolor="white", plot_bgcolor="white",
        margin=dict(l=0, r=0, t=45 if title else 0, b=0),
    )
    fig.show()


def image_size(pixels):
    return as_rgb_array(pixels).shape[:2]


def apply_sepia_matrix(image):
    matrix = np.array([
        [0.393, 0.769, 0.189],
        [0.349, 0.686, 0.168],
        [0.272, 0.534, 0.131],
    ])
    return np.clip(as_rgb_array(image) @ matrix.T, 0, 255).astype(np.uint8)


def apply_filter(pixels, R_new, G_new, B_new):
    new_pixels = []
    for row in pixels:
        new_row = []
        for pixel in row:
            r, g, b = (int(value) for value in pixel)
            filtered = R_new(r, g, b), G_new(r, g, b), B_new(r, g, b)
            new_row.append(tuple(max(0, min(255, int(value))) for value in filtered))
        new_pixels.append(new_row)
    return new_pixels


def show_filter_widget(image):
    if isinstance(image, (str, Path)):
        original = load_pixels(image)
        image_name = Path(image).name
    else:
        original = as_rgb_array(image)
        image_name = "Uploaded image"

    style = {"description_width": "initial"}
    def slider(value, description):
        return widgets.FloatSlider(
            value=value, min=0, max=1, step=0.01, description=description,
            style=style, continuous_update=False,
        )

    a1, a2, a3 = slider(1, "a1 (R→R):"), slider(0, "a2 (G→R):"), slider(0, "a3 (B→R):")
    b1, b2, b3 = slider(0, "b1 (R→G):"), slider(1, "b2 (G→G):"), slider(0, "b3 (B→G):")
    c1, c2, c3 = slider(0, "c1 (R→B):"), slider(0, "c2 (G→B):"), slider(1, "c3 (B→B):")

    controls = widgets.VBox([
        widgets.HTML("<h4>Red channel weights</h4>"), widgets.HBox([a1, a2, a3]),
        widgets.HTML("<h4>Green channel weights</h4>"), widgets.HBox([b1, b2, b3]),
        widgets.HTML("<h4>Blue channel weights</h4>"), widgets.HBox([c1, c2, c3]),
    ])
    plot_output = widgets.Output()

    def normalized(values, fallback):
        total = sum(values)
        return [value / total for value in values] if total else fallback

    def update_visualization(*_):
        matrix = np.array([
            normalized([a1.value, a2.value, a3.value], [1, 0, 0]),
            normalized([b1.value, b2.value, b3.value], [0, 1, 0]),
            normalized([c1.value, c2.value, c3.value], [0, 0, 1]),
        ])
        filtered = np.clip(original @ matrix.T, 0, 255).astype(np.uint8)
        fig = make_subplots(
            rows=1, cols=2, subplot_titles=("Original Image", "Filtered Image"),
            horizontal_spacing=0.02,
        )
        fig.add_trace(plotly_image_trace(original, "Original"), row=1, col=1)
        fig.add_trace(plotly_image_trace(filtered, "Filtered"), row=1, col=2)
        fig.update_xaxes(visible=False, showgrid=False, zeroline=False)
        fig.update_yaxes(visible=False, showgrid=False, zeroline=False)
        fig.update_layout(
            showlegend=False, height=500, paper_bgcolor="white", plot_bgcolor="white",
            margin=dict(l=0, r=0, t=45, b=0),
        )
        with plot_output:
            clear_output(wait=True)
            display(fig)

    sliders = [a1, a2, a3, b1, b2, b3, c1, c2, c3]
    for channel_slider in sliders:
        channel_slider.observe(update_visualization, names="value")

    display(widgets.VBox([
        widgets.HTML(f"<h2>Interactive RGB filter: {image_name}</h2>"),
        controls, plot_output,
    ]))
    update_visualization()

`load_pixels` reads an image file and hands back the grid of pixels. Let's look at a block M.

In [ ]:
pixels = load_pixels("https://raw.githubusercontent.com/math-124/fa26-code/main/homeworks/hw01/imgs/mblock_photo.jpeg")

show_pixels(pixels)
print("This image is", image_size(pixels)[0], "pixels tall and", image_size(pixels)[1], "pixels wide.")

**Above, if you hover over a pixel, you can see its red, green, and blue values!**

Now, let's apply a filter. We've written a helper function to show you an example of the kind of thing we're about to build. Run the cell below to apply a filter to the block M.

In [ ]:
sepia_block = apply_sepia_matrix(pixels)
show_pixels(sepia_block)

Now, your job is to apply what you've just learned about defining Python functions to create some filters of your own!

### Task 1

---

We just stated that an image filter is a function that takes a pixel's three color values and returns three new numbers. The first filter we'll look at is the **grayscale** filter, which turns an image into grayscale (black-and-white). A pixel is gray if its $R$, $G$, and $B$ values are all the same. Experiment with this on the Google color palette: enter `rgb(10, 10, 10)`, `rgb(50, 50, 50)`, and `rgb(200, 200, 200)`.

To convert a pixel to grayscale, its new red, green, and blue values must all be the same. To make this conversion, we'll set the new red, green, and blue values each to the **average** of the old red, green, and blue values:

$$
\begin{align*}
R_{\text{new}} &= \frac{1}{3} (\textcolor{red}{R_{\text{old}}} + \textcolor{green}{G_{\text{old}}} + \textcolor{blue}{B_{\text{old}}}) \\
G_{\text{new}} &= \frac{1}{3} (\textcolor{red}{R_{\text{old}}} + \textcolor{green}{G_{\text{old}}} + \textcolor{blue}{B_{\text{old}}}) \\
B_{\text{new}} &= \frac{1}{3} (\textcolor{red}{R_{\text{old}}} + \textcolor{green}{G_{\text{old}}} + \textcolor{blue}{B_{\text{old}}}) \\
\end{align*}
$$

**Below, complete the implementations of the functions `R_grayscale(r, g, b)`, `G_grayscale(r, g, b)`, and `B_grayscale(r, g, b)`. Each takes in a pixel's three components and returns the respective grayscale $R$, $G$, and $B$ values.** Example behavior is given below.

```python
>>> R_grayscale(50, 200, 14)
88

>>> G_grayscale(50, 200, 14)
88

>>> B_grayscale(50, 200, 14)
88
```

Don't round any of the converted values.

In [ ]:
def R_grayscale(r, g, b):
    """Return the R value of the grayscale version of this pixel."""
    return (r+g+b) / 3

def G_grayscale(r, g, b):
    """Return the G value of the grayscale version of this pixel."""
    return R_grayscale(r, g, b)

def B_grayscale(r, g, b):
    """Return the B value of the grayscale version of this pixel."""
    return R_grayscale(r, g, b)

### Task 2

---

We just stated that an image filter is a function that takes a pixel's three color values and returns three new numbers. The second filter we'll look at is the **sepia** filter, which applies a brown, nostalgic tint to an image. Unlike the grayscale filter, the converted red, green, and blue amounts in the new pixel are not all the same. Instead, each one is a different **weighted sum** of all three original color amounts.

To convert a pixel to sepia, we'll set the new red, green, and blue values to the following **weighted sums** of the old red, green, and blue values:

$$
\begin{align}
R_{\text{new}} &= 0.393\,\textcolor{red}{R_{\text{old}}} + 0.769\,\textcolor{green}{G_{\text{old}}} + 0.189\,\textcolor{blue}{B_{\text{old}}} \\
G_{\text{new}} &= 0.349\,\textcolor{red}{R_{\text{old}}} + 0.686\,\textcolor{green}{G_{\text{old}}} + 0.168\,\textcolor{blue}{B_{\text{old}}} \\
B_{\text{new}} &= 0.272\,\textcolor{red}{R_{\text{old}}} + 0.534\,\textcolor{green}{G_{\text{old}}} + 0.131\,\textcolor{blue}{B_{\text{old}}} \\
\end{align}
$$

**Below, complete the implementations of the functions `R_sepia(r, g, b)`, `G_sepia(r, g, b)`, and `B_sepia(r, g, b)`. Each takes in a pixel's three components and returns the respective sepia $R$, $G$, and $B$ values.** Example behavior is given below.

```python
>>> R_sepia(50, 200, 14)
176.096

>>> G_sepia(50, 200, 14)
157.002

>>> B_sepia(50, 200, 14)
122.23400000000001
```

Like before, don't round.

In [ ]:
def R_sepia(r, g, b):
    """Return the R value of the sepia version of this pixel."""
    # BEGIN SOLUTION
    R = 0.393 * r + 0.769 * g + 0.189 * b
    return R
    # END SOLUTION

def G_sepia(r, g, b):
    """Return the G value of the sepia version of this pixel."""
    # BEGIN SOLUTION
    G = 0.349 * r + 0.686 * g + 0.168 * b   
    return G
    # END SOLUTION

def B_sepia(r, g, b):
    """Return the B value of the sepia version of this pixel."""
    # BEGIN SOLUTION
    B = 0.272 * r + 0.534 * g + 0.131 * b
    return B
    # END SOLUTION

Let's visualize your grayscale and sepia filter on the block M sample image.

In [ ]:
gray_pixels = apply_filter(pixels, R_grayscale, G_grayscale, B_grayscale)
show_pixels(gray_pixels, title="Grayscale")

In [ ]:
sepia_pixels = apply_filter(pixels, R_sepia, G_sepia, B_sepia)
show_pixels(sepia_pixels, title="Sepia")

### Task 3

---

Now for the fun part: we'll apply all of this to an image of your own choosing! First, run the cell to select a photo of your own.

In [ ]:
from google.colab import files
from PIL import Image
import io

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one image.")

filename, data = next(iter(uploaded.items()))

# Verify that the uploaded file is actually an image
try:
    image = Image.open(io.BytesIO(data))
    image.verify()
except Exception:
    raise ValueError("Please upload a valid image file.")

# Load and display the uploaded file with the helpers defined above.
uploaded_pixels = load_pixels(filename)
show_pixels(uploaded_pixels, title=filename)

First, we'll apply the grayscale and sepia filters to your image.

In [ ]:
uploaded_gray_pixels = apply_filter(uploaded_pixels, R_grayscale, G_grayscale, B_grayscale)
show_pixels(uploaded_gray_pixels, title="Grayscale (custom photo)")

In [ ]:
uploaded_sepia_pixels = apply_filter(uploaded_pixels, R_sepia, G_sepia, B_sepia)
show_pixels(uploaded_sepia_pixels, title="Sepia (custom photo)")

Now, run the cell below to show a **custom filter creator**, where you can customize the filter that is applied to your image by changing the coefficients on $R_\text{old}$, $G_\text{old}$, and $B_\text{old}$ in the formulas for the new red, green, and blue amounts.

In [ ]:
show_filter_widget(filename)

## Finish line 🏁

To get credit for the work you did in this notebook, include screenshots of the following parts of your notebook as part of your PDF submission to Homework 1 on Pensive, specifically under Problem 6:

1. A screenshot of the code you wrote in Task 2, to implement the sepia filter.
2. A screenshot of the grayscale and sepia versions of the image you uploaded.
3. A screenshot of your custom filter and it applied to your image.